<a href="https://colab.research.google.com/github/CesarFloresQuintana/AI-Newsroom-Multi-Agent-Platform-for-Digital-Newspaper-Transformation/blob/main/agente_verificador_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instalacion de librerias
# !pip install -q -U langchain langchain-community langchain-core langchain-google-genai
# Instalamos la versión específica que contiene el código de agentes tradicional
#!pip install -q -U --force-reinstall langchain==0.1.20 langchain-community langchain-core langchain-google-genai


# 1. Borrado total de las librerías en conflicto
!pip uninstall -y langchain langchain-community langchain-core langchain-google-genai pydantic

# 2. Instalación de versiones equilibradas y modernas
#!pip install -q -U langchain-core langchain-google-genai langchain-community pydantic==2.10.0
!pip install -q -U google-generativeai langchain-core langchain-community google-genai

# 3. REINICIA EL ENTORNO: Ve a 'Entorno de ejecución' > 'Reiniciar sesión'


Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
Found existing installation: langchain-core 1.2.19
Uninstalling langchain-core-1.2.19:
  Successfully uninstalled langchain-core-1.2.19
Found existing installation: pydantic 2.12.5
Uninstalling pydantic-2.12.5:
  Successfully uninstalled pydantic-2.12.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [ ]:
#generación de entorno
import os
from google.colab import userdata

# Configuración de API Key
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')
    MI_LLAVE_API = userdata.get('GEMINI_API_KEY')
except:
    print("⚠️ Recuerda añadir 'GEMINI_API_KEY' en la sección de Secrets (llave) de Colab.")

In [ ]:
import json
import time
from google.colab import userdata

from google import genai
from google.genai import types
from langchain_core.tools import tool

from tenacity import retry, stop_after_attempt, wait_exponential
import requests

# Implementación de MPC
# mcp_archive (histórico del periódico con RAG)
# Este es el más importante para el periódico. servidor MCP local que consulte una base de datos vectorial.
# Carga tus datos: Sube los PDF o textos del periódico a un almacenamiento (Google Drive).
# Crea un Store Vectorial: Usa ChromaDB o FAISS en el Colab.
# Herramienta RAG: Define una función de Python que busque en esa base de datos y regístrala como una "tool" para Gemini.
@tool
def mcp_archive(query: str):
    """
    Consulta el archivo histórico global y local usando el Proyecto GDELT.
    Ideal para verificar eventos pasados, cifras mencionadas en prensa y cronologías.
    """
    base_url = "https://api.gdeltproject.org/api/v2/doc/doc"

    #return "Resultado real del archivo: El presupuesto en 2023 fue de 450.000€."
    # Parámetros de la consulta
    params = {
        "query": query,
        "mode": "artlist",          # Nos devuelve una lista de artículos
        "maxrecords": 5,            # Limitamos para no saturar de tokens a Gemini
        "format": "json",
        "sort": "rel"               # Ordenar por relevancia
    }

    try:
        response = requests.get(base_url, params=params, timeout=10)

        if response.status_code == 200:
            data = response.json()
            if "articles" in data:
                # Extraemos los títulos y fuentes para que el agente verifique
                resultados = []
                for art in data["articles"]:
                    resultados.append(f"Fuente: {art['source_country']} | Título: {art['title']} | Link: {art['url']}")

                contexto = "\n".join(resultados)
                return f"Hallazgos en el archivo histórico (GDELT):\n{contexto}"
            else:
                return "No se encontraron registros históricos específicos para esa consulta en GDELT."
        else:
            return f"Error al conectar con GDELT (Código: {response.status_code})."

    except Exception as e:
        return f"Error técnico consultando el archivo GDELT: {str(e)}"

# mcp_official y mcp_stats (APIs Externas)
# [mcp_official]: Para buscar en boletines oficiales.
# [mcp_stats]: Para contrastar cifras económicas.
# En lugar de programar cada API, utilizas servidores MCP ya existentes.
# Uso de Google Search MCP: uso del servidor oficial de Google Search para que el agente navegue por la web y busque el BOE o el INE en tiempo real.
# Sequential Thinking: Se configura para que el agente primero busque la cifra y luego la compare.
@tool
def mcp_official(entidad: str, fecha: str):
    """Busca en boletines oficiales vía API o Scraping."""
    # Simulación de llamada a API real (puedes usar requests aquí)
    return f"Boletín oficial de {entidad}: Cifra confirmada de 1.2M€."

# --- CLASE DEL AGENTE TRABAJADOR (WORKER) ---

class VerificadorAgent:
    def __init__(self, api_key):
        # MODIFICADO_AQUI: El agente ya tiene su propia configuración interna
        self.client = genai.Client(api_key=api_key)
        self.model_id = "models/gemini-2.5-flash"
        self.tools = [mcp_archive, mcp_official]
        self.funcs = {t.name: t for t in self.tools}

        self.config = types.GenerateContentConfig(
            tools=self.tools,
            system_instruction="""
            Eres el 'Agente Verificador' del Periódico Local.
            Tu misión: Detectar errores en datos (nombres, fechas, cifras). usando tus herramientas

            TAREAS:
            1. Identifica cifras o hechos clave.
            2. Si hay contradicciones obvias en el texto, usa UNA SOLA VEZ para confirmar
              - [mcp_archive]: Para buscar en el histórico del periódico.
              - [mcp_official]: Para buscar en boletines oficiales.
              - [mcp_stats]: Para contrastar cifras económicas.
            3. Si los datos coinciden, no uses herramientas.
            4. Evaluar la coherencia del relato.

            FORMATO DE RESPUESTA:
            - Estado: [APROBADO / REVISIÓN / RECHAZADO]
            - Hallazgos: Lista de puntos verificados.
            - Alertas: Si hay incoherencias (ej. fechas imposibles).
            - Sugerencia de fuentes: Qué servidores MCP usarías para confirmar.
            """
        )

    @retry(wait=wait_exponential(multiplier=2, min=10, max=60), stop=stop_after_attempt(5))
    def llamar_a_gemini(self, chat, mensaje):
        return chat.send_message(mensaje)

    # Este es el método que llamarán los otros agentes
    def ejecutar(self,instruccion: str):
        """
        Método de entrada para otros agentes.
        Recibe una instrucción y devuelve el veredicto final.
        """
        # El chat se crea internamente y se destruye al terminar la tarea (stateless)
        # o puedes mantenerlo en self si quieres que tenga memoria entre llamadas.
        chat = self.client.chats.create(model=self.model_id, config=self.config)
        #response = self.llamar_a_gemini(chat, instruccion)
        response = chat.send_message(instruccion)

        # Bucle interno de herramientas (el "cerebro" del trabajador)
        for _ in range(5):
            calls = [p.function_call for p in response.candidates[0].content.parts if p.function_call]
            if not calls:
                break

            tool_responses = []
            for call in calls:
                # Ejecución de la lógica de negocio
                result = self.funcs[call.name].invoke(call.args)
                tool_responses.append(types.Part.from_function_response(
                    name=call.name,
                    response={'result': result}
                ))

            response = chat.send_message(tool_responses)

        return response.text


In [ ]:
# --- EJEMPLO DE CÓMO LO LLAMARÍA OTRO AGENTE (ORQUESTADOR) ---

class EditorJefeAgent:
    def __init__(self,api_key):
        self.verificador = VerificadorAgent(api_key) # El editor jefe tiene a su cargo al verificador

    def procesar_noticia(self,titulo, contenido):
        print(f"👨‍💼 Editor Jefe: 'Verificador, revisa esto: {titulo}'")

        # El Editor llama al método 'ejecutar' del subordinado
        veredicto = self.verificador.ejecutar(f"Revisa esta noticia: {contenido}")

        return f"--- INFORME FINAL ---\nNOTICIA: {titulo}\nESTADO: {veredicto}"

# --- PRUEBA DEL SISTEMA MULTI-AGENTE ---
try:
    jefe = EditorJefeAgent(MI_LLAVE_API)
    noticia_test = {
        "titulo": "Gasto en Fuentes",
        "noticia": "El ayuntamiento de Madrid, durante 2024 dice que gastó 1M€ en gasto corriente, pero el archivo dice que fueron 2M€."
    }

    resultado = jefe.procesar_noticia(noticia_test["titulo"], noticia_test["noticia"])
    print("\n🚀 RESULTADO DEL SISTEMA:\n", resultado)

except Exception as e:
    print(f"❌ Error en la comunicación: {e}")

👨‍💼 Editor Jefe: 'Verificador, revisa esto: Gasto en Fuentes'

🚀 RESULTADO DEL SISTEMA:
 --- INFORME FINAL ---
NOTICIA: Gasto en Fuentes
ESTADO: - Estado: REVISIÓN
- Hallazgos: Se detecta una contradicción directa en las cifras de gasto corriente para el Ayuntamiento de Madrid durante 2024. El ayuntamiento declara 1M€, mientras que "el archivo" (presumiblemente un registro anterior o interno) indica 2M€.
- Alertas: Incoherencia clara en el monto del gasto corriente reportado por diferentes fuentes para el mismo período y concepto.
- Sugerencia de fuentes:
    *   [mcp_official]: Para verificar los datos oficiales de gasto corriente del Ayuntamiento de Madrid para el año 2024 en los boletines o informes de presupuestos.


In [ ]:
import google.genai as genai
from google.colab import userdata

# Recuperar cliente
MI_LLAVE_API = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=MI_LLAVE_API)

print("--- Nombres exactos de modelos disponibles ---")
try:
    # Listamos todos los modelos sin filtrar por atributos internos para evitar errores
    for m in client.models.list():
        # Solo imprimimos el nombre, que es lo que necesitamos para self.model_id
        print(f"✅ {m.name}")
except Exception as e:
    print(f"❌ Error al listar: {e}")

--- Nombres exactos de modelos disponibles ---
✅ models/gemini-2.5-flash
✅ models/gemini-2.5-pro
✅ models/gemini-2.0-flash
✅ models/gemini-2.0-flash-001
✅ models/gemini-2.0-flash-lite-001
✅ models/gemini-2.0-flash-lite
✅ models/gemini-2.5-flash-preview-tts
✅ models/gemini-2.5-pro-preview-tts
✅ models/gemma-3-1b-it
✅ models/gemma-3-4b-it
✅ models/gemma-3-12b-it
✅ models/gemma-3-27b-it
✅ models/gemma-3n-e4b-it
✅ models/gemma-3n-e2b-it
✅ models/gemini-flash-latest
✅ models/gemini-flash-lite-latest
✅ models/gemini-pro-latest
✅ models/gemini-2.5-flash-lite
✅ models/gemini-2.5-flash-image
✅ models/gemini-2.5-flash-lite-preview-09-2025
✅ models/gemini-3-pro-preview
✅ models/gemini-3-flash-preview
✅ models/gemini-3.1-pro-preview
✅ models/gemini-3.1-pro-preview-customtools
✅ models/gemini-3.1-flash-lite-preview
✅ models/gemini-3-pro-image-preview
✅ models/nano-banana-pro-preview
✅ models/gemini-3.1-flash-image-preview
✅ models/gemini-robotics-er-1.5-preview
✅ models/gemini-2.5-computer-use-prev